In [1]:
import cupy as cp
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
from tqdm import tqdm
import torchvision.utils as vutils
import torch
import math
import numpy as np
from numba import cuda
from safetensors.torch import save_file
# =====================================================================
# KERNEL 1: HINTON REPRESENTATION UPDATE (Contrastive Hebbian)
# =====================================================================
@cuda.jit
def hinton_update_kernel(W, x_pos, y_pos, x_neg, y_neg, scale_pos, scale_neg, lr):
    out_idx, in_idx = cuda.grid(2)
    if out_idx < W.shape[0] and in_idx < W.shape[1]:
        batch_size = x_pos.shape[0]
        delta = 0.0
        for b in range(batch_size):
            pos_term = scale_pos[b] * y_pos[b, out_idx] * x_pos[b, in_idx]
            neg_term = scale_neg[b] * y_neg[b, out_idx] * x_neg[b, in_idx]
            delta += (pos_term - neg_term)
        W[out_idx, in_idx] += (lr / batch_size) * delta

# =====================================================================
# KERNEL 2: KARL GENERATIVE UPDATE (Predictive Coding)
# =====================================================================
@cuda.jit
def karl_update_kernel(G, z_current, x_true, x_pred, lr):
    out_idx, in_idx = cuda.grid(2)
    if out_idx < G.shape[0] and in_idx < G.shape[1]:
        batch_size = z_current.shape[0]
        delta = 0.0
        for b in range(batch_size):
            error = x_true[b, out_idx] - x_pred[b, out_idx]
            delta += error * z_current[b, in_idx]
        G[out_idx, in_idx] += (lr / batch_size) * delta

In [2]:
class CUDAPFFLayer:
    def __init__(self, in_features, out_features, lr_rep=0.1, lr_gen=0.1):
        self.in_features = in_features
        self.out_features = out_features
        self.lr_rep = lr_rep
        self.lr_gen = lr_gen
        
        limit_w = np.sqrt(6 / (in_features + out_features))
        self.W = cp.random.uniform(-limit_w, limit_w, (out_features, in_features), dtype=cp.float32)
        self.G = cp.random.uniform(-limit_w, limit_w, (in_features, out_features), dtype=cp.float32)
        
        self.threads_per_block = (16, 16)
        self.blocks_W = (math.ceil(out_features / 16), math.ceil(in_features / 16))
        self.blocks_G = (math.ceil(in_features / 16), math.ceil(out_features / 16))

    def layer_norm(self, x):
        norms = cp.linalg.norm(x, axis=1, keepdims=True)
        return x / (norms + 1e-4)

    def forward_hinton(self, x_below):
        x_norm = self.layer_norm(x_below)
        pre_act = cp.matmul(x_norm, self.W.T)
        return cp.maximum(0, pre_act)
        
    def forward_karl(self, z_current):
        z_norm = self.layer_norm(z_current)
        return cp.matmul(z_norm, self.G.T)

    def train_representation(self, x_pos, x_neg, threshold=1.0):
        y_pos = self.forward_hinton(x_pos)
        y_neg = self.forward_hinton(x_neg)

        x_pos_norm = self.layer_norm(x_pos)
        x_neg_norm = self.layer_norm(x_neg)
        
        # FIX: Goodness is the SUM of squared activations, not the mean!
        g_pos = cp.sum(y_pos**2, axis=1)
        g_neg = cp.sum(y_neg**2, axis=1)
        
        p_pos = 1 / (1 + cp.exp(-(g_pos - threshold)))
        p_neg = 1 / (1 + cp.exp(-(g_neg - threshold)))
        
        scale_pos = 1.0 - p_pos
        scale_neg = p_neg
        
        hinton_update_kernel[self.blocks_W, self.threads_per_block](
            self.W, x_pos_norm, y_pos, x_neg_norm, y_neg, scale_pos, scale_neg, self.lr_rep
        )
        return y_pos, y_neg 

    def train_generative(self, z_current, x_true):
        x_pred = self.forward_karl(z_current)
        karl_update_kernel[self.blocks_G, self.threads_per_block](
            self.G, z_current, x_true, x_pred, self.lr_gen
        )
        mse = cp.mean((x_true - x_pred)**2)
        return mse, x_pred

In [3]:
# =====================================================================
# HINTON LABEL OVERLAY (CuPy Vectorized)
# =====================================================================
def overlay_y_on_x(x, y, num_classes=10):
    x_copy = x.copy()
    
    x_copy[:, :num_classes] = -1.0  
    
    batch_indices = cp.arange(x.shape[0])
    x_copy[batch_indices, y] = 2.0 
    
    return x_copy

# =====================================================================
# THE PURE CUDA NETWORK ORCHESTRATOR
# =====================================================================
class CUDAHintonKarlNetwork:
    def __init__(self):
        self.d_in = 784
        self.d_h1 = 512
        self.d_h2 = 512

        self.layer1 = CUDAPFFLayer(self.d_in, self.d_h1, lr_rep=0.1, lr_gen=0.1)
        self.layer2 = CUDAPFFLayer(self.d_h1, self.d_h2, lr_rep=0.1, lr_gen=0.1)

    def train_step(self, x_pos, x_neg):
        z1_pos, z1_neg = self.layer1.train_representation(x_pos, x_neg)
        z2_pos, z2_neg = self.layer2.train_representation(z1_pos, z1_neg)
        
        mse2, z1_pred = self.layer2.train_generative(z2_pos, z1_pos)
        mse1, x_pred = self.layer1.train_generative(z1_pos, x_pos)
        return x_pred

    def predict(self, x_raw):
        B = x_raw.shape[0]
        C = 10
        x_expanded = cp.repeat(x_raw, C, axis=0)
        labels = cp.tile(cp.arange(C), B)
        x_test = overlay_y_on_x(x_expanded, labels)
        
        z1 = self.layer1.forward_hinton(x_test)
        z2 = self.layer2.forward_hinton(z1)
        
        g1 = cp.sum(z1**2, axis=1)
        g2 = cp.sum(z2**2, axis=1)
        total_goodness = g1 + g2
        
        goodness_matrix = total_goodness.reshape(B, C)
        return cp.argmax(goodness_matrix, axis=1)

    def save_safetensors(self, filepath):
        """ Pulls weights from CuPy (VRAM) to CPU and saves as Safetensors """
        tensors = {
            "layer1.W": torch.from_numpy(self.layer1.W.get()),
            "layer1.G": torch.from_numpy(self.layer1.G.get()),
            "layer2.W": torch.from_numpy(self.layer2.W.get()),
            "layer2.G": torch.from_numpy(self.layer2.G.get())
        }
        save_file(tensors, filepath)
        print(f"\nModel weights successfully saved to {filepath}")

# =====================================================================
# EXECUTION SCRIPT
# =====================================================================
if __name__ == "__main__":
    transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.5,), (0.5,))
    ])

    print("Loading Dataset...")
    train_dataset = datasets.MNIST('./data', train=True, download=True, transform=transform)
    test_dataset = datasets.MNIST('./data', train=False, download=True, transform=transform)

    train_loader = DataLoader(train_dataset, batch_size=256, shuffle=True)
    test_loader = DataLoader(test_dataset, batch_size=256, shuffle=False)

    model = CUDAHintonKarlNetwork()
    print("Initiating Pure CUDA Forward-Forward Training...")

    EPOCHS = 20
    for epoch in range(EPOCHS):
        
        train_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Train]")
        for batch_idx, (data, target) in enumerate(train_bar):
            data_cp = cp.array(data.view(data.shape[0], -1).numpy())
            target_pos_cp = cp.array(target.numpy())
            
            target_neg_cp = cp.random.randint(0, 10, (data_cp.shape[0],))
            target_neg_cp = (target_pos_cp + cp.random.randint(1, 10, (data_cp.shape[0],))) % 10
            
            x_pos = overlay_y_on_x(data_cp, target_pos_cp)
            x_neg = overlay_y_on_x(data_cp, target_neg_cp)
            
            x_pred = model.train_step(x_pos, x_neg)
            
        print("\nGenerating Hallucinated Reconstructions...")
        orig_img = torch.from_numpy(data_cp[:16].get()).view(-1, 1, 28, 28)
        recon_img = torch.from_numpy(x_pred[:16].get()).view(-1, 1, 28, 28)
        
        combined = torch.cat([orig_img, recon_img], dim=0)
        img_path = f'reconstruction_epoch_{epoch+1}.png'
        vutils.save_image(combined, img_path, nrow=16, normalize=True)
        print(f"Saved Reconstructions to: {img_path}")
            
        print(f"Epoch {epoch+1} Complete. Testing classification...")
        
        correct = 0
        total = 0
        
        test_bar = tqdm(test_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Test]")
        for data, target in test_bar:
            data_cp = cp.array(data.view(data.shape[0], -1).numpy())
            target_cp = cp.array(target.numpy())
            
            preds = model.predict(data_cp)
            
            correct += int(cp.sum(preds == target_cp))
            total += target_cp.shape[0]
            
        print(f"\nValidation Accuracy: {100 * correct / total:.2f}%\n")

    # Save the custom CUDA weights securely after training completes
    model.save_safetensors("hinton_karl_pff.safetensors")

Loading Dataset...


100%|██████████| 9.91M/9.91M [00:00<00:00, 129MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 22.5MB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 140MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 12.0MB/s]


Initiating Pure CUDA Forward-Forward Training...


Epoch 1/20 [Train]: 100%|██████████| 235/235 [00:20<00:00, 11.59it/s]



Generating Hallucinated Reconstructions...
Saved Reconstructions to: reconstruction_epoch_1.png
Epoch 1 Complete. Testing classification...


...


Validation Accuracy: 54.20%



Epoch 20/20 [Train]: 100%|██████████| 235/235 [00:14<00:00, 15.89it/s]



Generating Hallucinated Reconstructions...
Saved Reconstructions to: reconstruction_epoch_20.png
Epoch 20 Complete. Testing classification...


Epoch 20/20 [Test]: 100%|██████████| 40/40 [00:02<00:00, 19.29it/s]


Validation Accuracy: 54.64%


Model weights successfully saved to hinton_karl_pff.safetensors
